In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Module Import

In [ ]:
!pip install -U bitsandbytes>=0.46.1
!pip install torch
!pip install trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 838.8/838.8 kB 25.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 37.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 14.5 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0


In [ ]:
import pandas as pd
import numpy as np
from  matplotlib import pylab as plt

import torch
from datasets import Dataset

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model

from trl import SFTTrainer
from transformers import TrainingArguments

In [ ]:
# This should return True if your NVIDIA GPU is recognized
print(torch.cuda.is_available())
# Optional: Check the name of your GPU
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

True
Tesla T4
Using device: cuda


# Data Import

Based on my analysis of `dataset.csv`, you have an excellent foundation for this project. The dataset contains 32,583 rows of financial text (news headlines, tweets, and market updates) mapped to perfectly balanced sentiment labels: **10,861 positive**, **10,861 neutral**, and **10,861 negative**.

Here is how we can apply the "Industry Standard" fine-tuning methodology to
this exact dataset to build a specialized Financial Sentiment LLM.

In [ ]:
# Load the dataset to inspect its structure and content
try:
    dataset = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/NLP & LLMs/Finance News Sentiments (LLM/dataset.csv")

    # Get basic info
    info_buf = []
    import io
    buf = io.StringIO()
    dataset.info(buf=buf)
    info_str = buf.getvalue()

    # Get the first few rows
    head_str = dataset.head().to_string()

    print("--- INFO ---")
    print(info_str)
    print("\n--- HEAD ---")
    print(head_str)
except Exception as e:
    print(f"Error loading dataset: {e}")

--- INFO ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32583 entries, 0 to 32582
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   sentiment  32583 non-null  object
 1   text       32569 non-null  object
dtypes: object(2)
memory usage: 509.2+ KB


--- HEAD ---
  sentiment                                                                                                                  text
0  positive  All banks, lending institutions may allow a three-month moratorium on all loans, says RBI chief Shaktikanta Das ami…
1   neutral                                                                                                 Not so fast, Drake...
2  positive            FNF - ong  19.43. Trailing Stop  21.04 from 6 prior Stops of 19.71, 19.52, 18.70, 18.65, 18.07 and 17.87 -
3  positive                                      Dow opens down almost 500 points after China hits back with new round of tariffs
4  positive     

#Step 1: Data Preparation & Prompt Engineering

To train the LLM, we must convert your tabular dataset.csv into an instruction-response format. We need to drop the few rows with missing text and format the rest so the model understands the task.

In [ ]:
# Load and format the data
df = dataset.copy().dropna(subset=['text'])

def generate_prompt(row):
    return f"""Below is a financial text. Analyze it and classify the sentiment as positive, neutral, or negative.

### Text:
{row['text']}

### Sentiment:
{row['sentiment']}"""

# FIX: Save the formatted string to a column named 'text', not 'prompt'
df['text'] = df.apply(generate_prompt, axis=1)

# FIX: Create the dataset using the 'text' column
hf_dataset = Dataset.from_pandas(df[['text']])

# Create a 90/10 Train/Validation Split
split_dataset = hf_dataset.train_test_split(test_size=0.1, seed=42)
train_data = split_dataset['train']
eval_data = split_dataset['test']

print(f"Training samples: {len(train_data)}")
print(f"Validation samples: {len(eval_data)}")

Training samples: 29312
Validation samples: 3257


# Step 2: Model Setup & LoRA Injection

We will load a pre-trained open-weights model in 4-bit precision to save compute, and then inject the LoRA adapters into the attention modules.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model

# CHANGE THIS LINE: Use Mistral instead of LLaMA
model_id = "mistralai/Mistral-7B-v0.1"

# You don't even need the token for Mistral, but it doesn't hurt to keep it
hf_token = ""

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype="float16",
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto", # <-- Auto run on GPU
    quantization_config=bnb_config,
    token=hf_token
)

# For Mistral, we need to add a padding token if it doesn't have one
tokenizer = AutoTokenizer.from_pretrained(model_id, token=hf_token)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, peft_config)
print("Mistral Model loaded successfully! Ready for fine-tuning.")

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/25.1k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

...

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer.json:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

Mistral Model loaded successfully! Ready for fine-tuning.


# Step 3: Supervised Fine-Tuning (SFT)

Finally, you pass your formatted dataset.csv data through the model. The base LLaMA weights stay frozen, and backpropagation only updates the small LoRA matrices based on how accurately it predicts "positive", "neutral", or "negative".

In [ ]:
from trl import SFTTrainer, SFTConfig

# FIX: Change max_seq_length to max_length
training_args = SFTConfig(
    output_dir="./financial-sentiment-model",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    max_steps=500,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",

    dataset_text_field="text",

    # -- RENAMED ARGUMENT --
    max_length=256
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_data,
    eval_dataset=eval_data,
    args=training_args,
    processing_class=tokenizer
)

trainer.train()

Adding EOS to train dataset:   0%|          | 0/29312 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/29312 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/29312 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/3257 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/3257 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/3257 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
50,1.275697,1.274638,1.293221,53089.000000,0.754121
100,1.170858,1.154713,1.163589,106689.000000,0.772422


# Validation

In [ ]:
# Put the model in evaluation mode
model.eval()

# A hypothetical unseen text relevant to your domain
text_to_predict = "Vinamilk reports a significant year-over-year increase in Q3 Net Income, driven by highly optimized CapEx strategies."

# Format the input exactly as the model was trained, leaving the sentiment blank
prompt = f"""Below is a financial text. Analyze it and classify the sentiment as positive, neutral, or negative.

### Text:
{text_to_predict}

### Sentiment:
"""

# Tokenize and push to GPU
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

# Generate the prediction
outputs = model.generate(**inputs, max_new_tokens=10)

# Decode and print the result
result = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(result)